In [1]:
!pip install torch --quiet

In [2]:
!pip install plotly

In [3]:
!pip install yfinance pandas numpy torch matplotlib gradio

In [4]:
!pip install ccxt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.8/151.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 16.3 MB/s eta 0:00:00


In [5]:
import torch
import torch.nn as nn

In [6]:
import numpy as np

try:
    import ccxt
except:
    import subprocess
    subprocess.check_call(["pip", "install", "ccxt"])
    import ccxt

In [7]:
def init_exchange():
    try:
        import ccxt
        exchange = ccxt.kraken({
            'enableRateLimit': True
        })
        return exchange
    except Exception as e:
        print("Exchange error:", e)
        return None

exchange = init_exchange()

In [8]:
import numpy as np
import ccxt

# =========================
# EXCHANGE (FIXED)
# =========================
def init_exchange():
    try:
        exchange = ccxt.kraken({
            'enableRateLimit': True
        })
        return exchange
    except Exception as e:
        print("Exchange error:", e)
        return None

exchange = init_exchange()

# =========================
# SYMBOL MAP
# =========================
def map_to_crypto(ticker):
    ticker = ticker.upper()

    mapping = {
        "AAPL": "BTC/USD",
        "TSLA": "ETH/USD",
        "MSFT": "BTC/USD"
    }

    if "/" in ticker:
        return ticker.replace("USDT", "USD")

    return mapping.get(ticker, "BTC/USD")

# =========================
# FETCH ORDERBOOK
# =========================
def get_orderbook(symbol="BTC/USD", depth=20):

    if exchange is None:
        return None, None

    try:
        ob = exchange.fetch_order_book(symbol, limit=depth)

        bids = np.array(ob['bids'], dtype=float)
        asks = np.array(ob['asks'], dtype=float)

        if bids.size == 0 or asks.size == 0:
            return None, None

        return bids, asks

    except Exception as e:
        print("Orderbook error:", e)
        return None, None

In [9]:
symbol = "BTC/USDT"
bids, asks = get_orderbook(symbol)

print("Bids sample:", bids[:3] if bids is not None else None)
print("Asks sample:", asks[:3] if asks is not None else None)

Bids sample: [[6.38310000e+04 1.00000000e-03 1.78133699e+09]
 [6.38279000e+04 8.30000000e-02 1.78133699e+09]
 [6.38278000e+04 1.10000000e-02 1.78133699e+09]]
Asks sample: [[6.38367000e+04 1.85000000e-01 1.78133699e+09]
 [6.38405000e+04 2.30000000e-02 1.78133699e+09]
 [6.38407000e+04 3.20000000e-02 1.78133699e+09]]


In [10]:
def compute_orderbook_signal(bids, asks):
    if bids is None or asks is None:
        return 0

    bid_volume = bids[:,1].sum()
    ask_volume = asks[:,1].sum()

    imbalance = (bid_volume - ask_volume) / (bid_volume + ask_volume + 1e-9)

    # Signal:
    if imbalance > 0.1:
        return 1   # bullish
    elif imbalance < -0.1:
        return -1  # bearish
    else:
        return 0

In [11]:
def compute_orderbook_metrics(bids, asks):
    import numpy as np

    if bids is None or asks is None:
        return None

    bids = np.array(bids, dtype=float)
    asks = np.array(asks, dtype=float)

    best_bid = bids[0, 0]
    best_ask = asks[0, 0]

    spread = best_ask - best_bid
    mid_price = (best_bid + best_ask) / 2

    bid_volume = bids[:,1].sum()
    ask_volume = asks[:,1].sum()

    imbalance = (bid_volume - ask_volume) / (bid_volume + ask_volume + 1e-9)

    return {
        "spread": spread,
        "mid_price": mid_price,
        "bid_volume": bid_volume,
        "ask_volume": ask_volume,
        "imbalance": imbalance
    }

In [12]:
def is_crypto(ticker):
    return "/" in ticker or ticker.upper() in ["BTC","ETH","BTC/USD","ETH/USD"]

In [13]:
FEATURE_COLS = [
    'return','momentum','volatility',
    'ema_fast','ema_slow','rsi',
    'macd','signal','volume_z'
]

TOP_STOCKS = [
    "AAPL","MSFT","GOOGL","AMZN","TSLA","NVDA","META",
    "NFLX","AMD","INTC",
    "RELIANCE.NS","TCS.NS","INFY.NS","HDFCBANK.NS","ICICIBANK.NS"
]

In [14]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = -delta.clip(upper=0).rolling(period).mean()
    rs = gain / (loss + 1e-9)
    return 100 - (100 / (1 + rs))

In [15]:
import yfinance as yf
import pandas as pd
import numpy as np

In [16]:
def validate_ticker(ticker):
    import yfinance as yf

    try:
        df = yf.download(ticker, period="5d", progress=False)

        if df is None or len(df) == 0:
            return False

        return True
    except:
        return False

In [17]:
POPULAR_STOCKS = [
    "AAPL", "MSFT", "GOOGL", "TSLA", "AMZN",
    "NVDA", "META",
    "RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS"
]

In [18]:
def get_data(ticker="AAPL"):
    df = yf.download(ticker, period="5y", progress=False)

    df = df[['Open','High','Low','Close','Volume']].copy()

    df['return'] = df['Close'].pct_change()
    df['momentum'] = df['Close'] / df['Close'].shift(10) - 1
    df['volatility'] = df['return'].rolling(10).std()

    df['ema_fast'] = df['Close'].ewm(span=10).mean()
    df['ema_slow'] = df['Close'].ewm(span=50).mean()

    df['rsi'] = compute_rsi(df['Close'])

    # NEW FEATURES
    df['macd'] = df['Close'].ewm(span=12).mean() - df['Close'].ewm(span=26).mean()
    df['signal'] = df['macd'].ewm(span=9).mean()

    df['volume_z'] = (df['Volume'] - df['Volume'].rolling(20).mean()) / (
        df['Volume'].rolling(20).std() + 1e-9
    )

    df = df.dropna()

    return df

In [19]:
def map_to_crypto(ticker):
    mapping = {
        "AAPL": "BTC/USDT",
        "TSLA": "ETH/USDT"
    }
    return mapping.get(ticker, "BTC/USDT")

In [20]:
def get_live_price(ticker):
    try:
        df = yf.download(ticker, period="1d", interval="1m", progress=False)

        if df is None or len(df) == 0:
            return None, None

        # 🔥 FORCE CLEAN 1D ARRAY
        close = df['Close']

        # Handle multi-index / dataframe issues
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]

        prices = close.astype(float).values.flatten()

        latest_price = float(prices[-1])

        return latest_price, prices

    except Exception as e:
        print("Live Error:", e)
        return None, None

In [21]:
def normalize(df):
    df = df.copy()

    df[FEATURE_COLS] = (df[FEATURE_COLS] - df[FEATURE_COLS].mean()) / (df[FEATURE_COLS].std() + 1e-9)


    return df

In [22]:
def create_sequences(df, seq_len=20):
    features = df[FEATURE_COLS].values

    X, y = [], []

    for i in range(len(features) - seq_len):
        X.append(features[i:i+seq_len])
        y.append(features[i+seq_len][0])

    X = np.array(X)
    y = np.array(y)

    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

In [23]:
def compute_drawdown(portfolio):
    p = np.array(portfolio)

    peak = np.maximum.accumulate(p)
    drawdown = (p - peak) / peak

    return drawdown

In [24]:
def benchmark_strategies(prices, initial_capital=10000):
    prices = np.array(prices)

    # Buy & Hold
    shares = initial_capital / prices[0]
    buy_hold = shares * prices

    # Sell (inverse proxy)
    sell = initial_capital * (prices[0] / prices)

    return buy_hold, sell

In [25]:
class TCN(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.conv1 = nn.Conv1d(input_size, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(32, 32, kernel_size=3, padding=1)

        self.relu = nn.ReLU()
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)

        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))

        x = x[:, :, -1]

        return self.fc(x)

In [26]:
def train_model(df):
    X, y = create_sequences(df)

    # Reduce size for speed
    X = X[:500]
    y = y[:500]
    print("Feature shape:", X.shape)
    model = TCN(input_size=X.shape[2])

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()

    for epoch in range(3):
        optimizer.zero_grad()
        output = model(X).squeeze()
        loss = loss_fn(output, y)
        loss.backward()
        optimizer.step()

    return model

In [27]:
def compute_factor_importance(model, df):
    model.eval()

    feature_cols = FEATURE_COLS
    base = df[feature_cols].values[-20:]  # last window

    base_tensor = torch.tensor(base, dtype=torch.float32).unsqueeze(0)

    base_pred = model(base_tensor).item()

    importance = {}

    for i, col in enumerate(feature_cols):
        modified = base.copy()

        # perturb feature
        modified[:, i] += 0.01

        mod_tensor = torch.tensor(modified, dtype=torch.float32).unsqueeze(0)

        new_pred = model(mod_tensor).item()

        importance[col] = abs(new_pred - base_pred)

    return importance

In [28]:
def backtest(df, model, ticker, seq_len=20):
    import numpy as np
    import torch

    # 🔥 Ensure feature consistency
    features = df[FEATURE_COLS].values.astype(float)

    # 🔥 FIX: Clean price array (same fix as live issue)
    close = df['Close']

    if isinstance(close, pd.DataFrame):  # handle multi-column case
        close = close.iloc[:, 0]

    prices = close.astype(float).values.flatten()

    capital = 10000.0
    position = 0.0
    entry_price = 0.0

    portfolio = []
    trades = []

    # 🔥 DETERMINE IF CRYPTO
    use_orderbook = is_crypto(ticker)

    if use_orderbook:
        symbol = map_to_crypto(ticker)
        bids, asks = get_orderbook(symbol)
        ob_signal = compute_orderbook_signal(bids, asks)
    else:
        ob_signal = None

    for i in range(seq_len, len(prices)):

        # 🔥 Build input safely
        seq = torch.tensor(features[i-seq_len:i], dtype=torch.float32).unsqueeze(0)

        pred = model(seq).item()

        # 🔥 Safety check
        if not np.isfinite(pred):
            continue

        price = float(prices[i])

        if price <= 0 or not np.isfinite(price):
            continue

        # ======================
        # ENTRY LOGIC
        # ======================
        if pred > 0.002 and position == 0:
            if not use_orderbook or ob_signal == 1:
                position = capital / price
                entry_price = price
                capital = 0.0

        # ======================
        # EXIT LOGIC
        # ======================
        elif pred < -0.002 and position > 0:
            if not use_orderbook or ob_signal == -1:
                exit_value = position * price

                trade_return = (price - entry_price) / (entry_price + 1e-9)
                trades.append(float(trade_return))

                capital = exit_value
                position = 0.0

        # ======================
        # PORTFOLIO VALUE
        # ======================
        value = capital + position * price

        if np.isfinite(value):
            portfolio.append(float(value))

    # 🔥 Final safety cleanup
    portfolio = np.array(portfolio, dtype=float)
    portfolio = portfolio[np.isfinite(portfolio)]

    symbol = map_to_crypto(ticker)
    bids, asks = get_orderbook(symbol)
    ob_signal = compute_orderbook_signal(bids, asks)

    return portfolio.tolist(), trades, prices[seq_len:]

In [29]:
def evaluate(portfolio):
    if len(portfolio) < 10:
        return 0

    p = np.array(portfolio, dtype=float)

    # Remove bad values
    p = p[np.isfinite(p)]

    if len(p) < 10:
        return 0

    returns = np.diff(p) / (p[:-1] + 1e-9)

    returns = returns[np.isfinite(returns)]

    if len(returns) < 2:
        return 0

    sharpe = np.mean(returns) / (np.std(returns) + 1e-9) * np.sqrt(252)

    return sharpe

In [30]:
def compute_metrics(portfolio, trades, initial_capital=10000):
    import numpy as np

    p = np.array(portfolio, dtype=float)

    # 🔥 Safety
    p = p[np.isfinite(p)]
    if len(p) < 5:
        return {}

    # =========================
    # RETURNS
    # =========================
    returns = np.diff(p) / (p[:-1] + 1e-9)
    returns = returns[np.isfinite(returns)]

    # =========================
    # CORE METRICS
    # =========================
    total_return = (p[-1] - initial_capital) / initial_capital
    profit = p[-1] - initial_capital

    sharpe = np.mean(returns) / (np.std(returns) + 1e-9) * np.sqrt(252)

    # Volatility
    volatility = np.std(returns) * np.sqrt(252)

    # Max Drawdown
    peak = np.maximum.accumulate(p)
    drawdown = (p - peak) / peak
    mdd = np.min(drawdown)

    # =========================
    # TRADE METRICS
    # =========================
    trades = np.array(trades, dtype=float)

    if len(trades) > 0:
        winrate = np.mean(trades > 0)
        avg_trade = np.mean(trades)
    else:
        winrate = 0
        avg_trade = 0

    return {
        "Sharpe": sharpe,
        "Total Return": total_return,
        "Profit": profit,
        "Max Drawdown": mdd,
        "Win Rate": winrate,
        "Num Trades": len(trades),
        "Avg Trade Return": avg_trade,
        "Volatility": volatility
    }

In [31]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_quant_dashboard(portfolio, prices, ticker):

    portfolio = np.array(portfolio)
    prices = np.array(prices)

    # Buy & Hold
    shares = 10000 / prices[0]
    buy_hold = shares * prices

    # Drawdown
    peak = np.maximum.accumulate(portfolio)
    drawdown = (portfolio - peak) / peak

    # Create subplots
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=("Equity Curve vs Benchmark", "Drawdown")
    )

    # 🔵 Strategy
    fig.add_trace(go.Scatter(
        y=portfolio,
        mode='lines',
        name='Strategy',
        line=dict(width=3)
    ), row=1, col=1)

    # 🟡 Buy & Hold
    fig.add_trace(go.Scatter(
        y=buy_hold,
        mode='lines',
        name='Buy & Hold',
        line=dict(dash='dash')
    ), row=1, col=1)

    # 🔴 Drawdown
    fig.add_trace(go.Scatter(
        y=drawdown,
        mode='lines',
        name='Drawdown',
        fill='tozeroy'
    ), row=2, col=1)

    # 🎨 PROFESSIONAL STYLE
    fig.update_layout(
        template="plotly_dark",
        title=f"{ticker} Quant Strategy Dashboard",
        hovermode="x unified",  # 🔥 enables pointer tracking
        height=700
    )

    return fig

In [32]:
def plot_live_price(prices, ticker):
    import plotly.graph_objects as go
    import numpy as np

    prices = np.array(prices).astype(float).flatten()

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=list(range(len(prices))),  # ✅ FIX X-AXIS
        y=prices,
        mode='lines',
        name='Live Price',
        line=dict(width=2)
    ))

    fig.update_layout(
        template="plotly_dark",
        title=f"{ticker} Live Price",
        hovermode="x unified",
        xaxis_title="Time Index",
        yaxis_title="Price"
    )

    return fig

In [33]:
def plot_factor_importance(importance_dict):
    import plotly.graph_objects as go

    features = list(importance_dict.keys())
    values = list(importance_dict.values())

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=features,
        y=values
    ))

    fig.update_layout(
        template="plotly_dark",
        title="Factor Attribution (Feature Importance)",
        xaxis_tickangle=-45
    )

    return fig

In [34]:
def walk_forward_validation(df, ticker, train_size=0.6, step_size=0.1):

    n = len(df)
    train_end = int(n * train_size)
    step = int(n * step_size)

    all_portfolios = []
    all_trades = []

    i = train_end

    while i < n - step:

        train_df = df.iloc[:i].copy()
        test_df = df.iloc[i:i+step].copy()

        # 🔥 Train model on past data only
        model = train_model(train_df)

        # 🔥 Test on future slice
        portfolio, trades, _ = backtest(test_df, model, ticker)

        all_portfolios.extend(portfolio)
        all_trades.extend(trades)

        i += step

    return all_portfolios, all_trades

In [35]:
def plot_orderbook(bids, asks):
    import numpy as np
    import plotly.graph_objects as go

    bids = np.array(bids, dtype=float)
    asks = np.array(asks, dtype=float)

    # 🔥 SORT DATA
    bids = bids[bids[:,0].argsort()[::-1]]   # descending
    asks = asks[asks[:,0].argsort()]         # ascending

    # 🔥 CUMULATIVE VOLUME
    bid_prices = bids[:,0]
    bid_volume = np.cumsum(bids[:,1])

    ask_prices = asks[:,0]
    ask_volume = np.cumsum(asks[:,1])

    fig = go.Figure()

    # 🟢 BID DEPTH
    fig.add_trace(go.Scatter(
        x=bid_prices,
        y=bid_volume,
        mode='lines',
        name='Bids',
        line=dict(color='green', width=3),
        fill='tozeroy'
    ))

    # 🔴 ASK DEPTH
    fig.add_trace(go.Scatter(
        x=ask_prices,
        y=ask_volume,
        mode='lines',
        name='Asks',
        line=dict(color='red', width=3),
        fill='tozeroy'
    ))

    fig.update_layout(
        template="plotly_dark",
        title="Order Book Depth (Cumulative)",
        xaxis_title="Price",
        yaxis_title="Cumulative Volume",
        hovermode="x unified"
    )

    return fig

In [36]:
def run_single_stock(ticker):
    try:
        df = get_data(ticker)
        df = normalize(df)

        model = train_model(df)
        # 🔥 USE WALK-FORWARD
        use_wfv = True

        if use_wfv:
            portfolio, trades = walk_forward_validation(df, ticker)
            prices = df['Close'].values[-len(portfolio):]
        else:
            portfolio, trades, prices = backtest(df, model, ticker)

        metrics = compute_metrics(portfolio, trades)

        return {
            "ticker": ticker,
            "portfolio": portfolio,
            "prices": prices,
            "metrics": metrics
        }

    except Exception as e:
        return {
            "ticker": ticker,
            "error": str(e)
        }


import matplotlib.pyplot as plt

def compare_stocks(tickers):
    fig = go.Figure()

    results = []

    for ticker in tickers:
        res = run_single_stock(ticker)

        if "error" in res:
            continue

        results.append(res)

        fig.add_trace(go.Scatter(
            y=res["portfolio"],
            mode='lines',
            name=ticker
        ))

    fig.update_layout(
        template="plotly_dark",
        title="Multi-Asset Strategy Comparison",
        hovermode="x unified"
    )

    return results, fig

def batch_backtest(stock_list):
    results = []

    for ticker in stock_list:
        res = run_single_stock(ticker)

        if "error" in res:
            continue

        m = res["metrics"]

        results.append({
            "Ticker": ticker,
            "Sharpe": round(m["Sharpe"],3),
            "Return (%)": round(m["Total Return"]*100,2),
            "Profit": round(m["Profit"],2),
            "MDD (%)": round(m["Max Drawdown"]*100,2),
            "Win Rate (%)": round(m["Win Rate"]*100,2),
            "Trades": m["Num Trades"]
        })

    df_results = pd.DataFrame(results)

    # 🔥 Sort by Sharpe (best first)
    df_results = df_results.sort_values(by="Sharpe", ascending=False)

    return df_results


def run_multi_mode(mode, ticker_input):
    try:
        ticker = ticker_input.split(",")[0].strip().upper()

        # 🔵 LIVE MODE
        if mode == "Live":

            price, prices = get_live_price(ticker)

            if price is None:
                return "Error fetching live data", None

            fig = plot_live_price(prices, ticker)
            price = float(price) if price is not None else 0

            return f"{ticker} Live Price: {price:.2f}", fig

        # 🟢 SINGLE MODE
        elif mode == "Single":
            result = run_single_stock(ticker)

            if "error" in result:
                return result["error"], None

            m = result["metrics"]

            fig = plot_quant_dashboard(
                result["portfolio"],
                result["prices"],
                ticker
            )


            # 🔥 Factor importance
            df = get_data(ticker)
            df = normalize(df)

            model = train_model(df)
            importance = compute_factor_importance(model, df)

           # fig = plot_factor_importance(importance)

            text = f"""
📊 TICKER: {ticker}

💰 Profit: {m['Profit']:.2f}
📈 Total Return: {m['Total Return']*100:.2f}%

⚡ Sharpe Ratio: {m['Sharpe']:.3f}
📉 Max Drawdown: {m['Max Drawdown']*100:.2f}%
📊 Volatility: {m['Volatility']*100:.2f}%

🎯 Win Rate: {m['Win Rate']*100:.2f}%
🔁 Number of Trades: {m['Num Trades']}
📌 Avg Trade Return: {m['Avg Trade Return']*100:.2f}%
"""

            return text, fig

        # 🟡 COMPARE MODE (unchanged)
        elif mode == "Compare":
            tickers = [t.strip().upper() for t in ticker_input.split(",")]
            results, fig = compare_stocks(tickers)

            text = "\n".join([
                f"{r['ticker']} → Sharpe: {r['metrics']['Sharpe']:.2f}"
                for r in results
            ])

            return text, fig

        # 🔴 BATCH MODE
        elif mode == "Batch":
            df_results = batch_backtest(TOP_STOCKS)

            fig = plot_batch_results(df_results)

            text = df_results.to_string(index=False)

            return text, fig


        elif mode == "OrderBook":

            ticker = ticker_input.strip().upper()
            symbol = map_to_crypto(ticker)

            bids, asks = get_orderbook(symbol)

            if bids is None or asks is None:
                return "OrderBook not available", None

            # 🔥 Compute metrics
            metrics = compute_orderbook_metrics(bids, asks)

            fig = plot_orderbook(bids, asks)

            text = f"""
        Symbol: {symbol}

        Mid Price: {metrics['mid_price']:.2f}
        Spread: {metrics['spread']:.2f}

        Bid Volume: {metrics['bid_volume']:.2f}
        Ask Volume: {metrics['ask_volume']:.2f}

        Imbalance: {metrics['imbalance']:.3f}
        """

            return text, fig
        elif mode == "WalkForward":

            df = get_data(ticker)
            df = normalize(df)

            portfolio, trades = walk_forward_validation(df, ticker)

            metrics = compute_metrics(portfolio, trades)

            fig = plot_quant_dashboard(
                portfolio,
                df['Close'].values[-len(portfolio):],
                ticker
            )

            text = f"""
        🚶 WALK-FORWARD VALIDATION

        Ticker: {ticker}

        Sharpe: {metrics['Sharpe']:.3f}
        Return: {metrics['Total Return']*100:.2f}%
        MDD: {metrics['Max Drawdown']*100:.2f}%
        Win Rate: {metrics['Win Rate']*100:.2f}%
        """

            return text, fig
    except Exception as e:
        return f"ERROR: {str(e)}", None




In [37]:
def plot_batch_results(df_results):
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=df_results["Ticker"],
        y=df_results["Sharpe"],
        name="Sharpe"
    ))

    fig.update_layout(
        template="plotly_dark",
        title="Sharpe Ratio Comparison",
        hovermode="x"
    )

    return fig

In [38]:
def run_live_mode(ticker):
    price, prices = get_live_price(ticker)

    if price is None:
        return "Error fetching live data", None

    fig = plot_live_price(prices, ticker)

    return f"{ticker} Live Price: {float(price):.2f}", fig

    print("Shape:", prices.shape)
    print("Sample:", prices[:5])

In [ ]:
# ================================
# 🔥 SAFE IMPORT
# ================================
try:
    import gradio as gr
except:
    import subprocess
    subprocess.check_call(["pip", "install", "gradio==4.44.0"])
    import gradio as gr
# ================================
# 🎨 CUSTOM CSS (DARK BLUE + ORANGE)
# ================================


custom_css = """
/* =========================
   🌌 GLOBAL BACKGROUND
========================= */
body {
    background: radial-gradient(circle at top, #0a0a0f, #020204 70%);
    color: #ccffe6;
    font-family: 'Segoe UI', sans-serif;
}

/* =========================
   🧠 TITLE (NEON GREEN)
========================= */
h1 {
    text-align: center;
    font-size: 42px;
    font-weight: 700;
    background: linear-gradient(90deg, #00ff9c, #00c3ff);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

/* =========================
   📦 MAIN SECTIONS
========================= */
.section {
    background: linear-gradient(145deg, rgba(0,255,156,0.12), rgba(0,40,60,0.25));
    border-radius: 16px;
    padding: 18px;
    margin-bottom: 18px;
    backdrop-filter: blur(14px);
    box-shadow: 0 0 30px rgba(0,255,156,0.15);
    transition: all 0.3s ease-in-out;
    border: 1px solid rgba(0,255,156,0.2);
}

.section:hover {
    transform: translateY(-3px);
    box-shadow: 0 0 40px rgba(0,255,156,0.35);
}

/* =========================
   📊 SUB PANELS
========================= */
.subsection {
    background: rgba(0,255,156,0.08);
    border-radius: 12px;
    padding: 12px;
    border: 1px solid rgba(0,255,156,0.15);
}

/* =========================
   🔘 BUTTON STYLE
========================= */
button {
    border-radius: 12px !important;
    font-weight: 600 !important;
    background: linear-gradient(135deg, #00ff9c, #00c3ff) !important;
    color: black !important;
    border: none !important;
    box-shadow: 0 0 15px rgba(0,255,156,0.6);
    transition: all 0.25s ease;
}

button:hover {
    transform: scale(1.05);
    box-shadow: 0 0 25px rgba(0,255,156,0.9);
}

/* =========================
   📥 INPUT BOXES
========================= */
textarea, input {
    background: rgba(20, 35, 30, 0.8) !important;
    color: #ccffe6 !important;
    border: 1px solid rgba(0,255,156,0.3) !important;
    border-radius: 10px !important;
}

/* =========================
   📊 PLOT AREA GLOW
========================= */
.plot-container {
    border-radius: 12px;
    box-shadow: 0 0 25px rgba(0,255,156,0.2);
}

/* =========================
   🧾 TEXTBOX OUTPUT (LLM/METRICS)
========================= */
textarea {
    font-family: monospace;
    font-size: 14px;
}

/* =========================
   🌈 SCROLLBAR (OPTIONAL)
========================= */
::-webkit-scrollbar {
    width: 8px;
}
::-webkit-scrollbar-thumb {
    background: #00ff9c;
    border-radius: 10px;
}
"""

# ================================
# ⚡ LOADING WRAPPER
# ================================
def run_with_loading(mode, ticker):
    import time
    time.sleep(0.4)
    return run_multi_mode(mode, ticker)

# ================================
# 🧊 BUILD UI (NEW LAYOUT)
# ================================
with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:

    # 🔥 HEADER
    gr.Markdown(
        """
        # 📊 QUANT TRADING TERMINAL
        ### AI Strategy • Order Book • Live Markets
        """,
    )

    # ================================
    # 🔹 TOP PANEL (CONTROLS)
    # ================================
    with gr.Row():

        mode = gr.Radio(
            ["Live","Single","Compare","Batch","OrderBook","WalkForward"],
            label="Mode",
            value="Single",
            scale=1
        )

        ticker = gr.Textbox(
            value="AAPL",
            label="Ticker(s)",
            placeholder="AAPL, TSLA, BTC/USDT",
            scale=2
        )

        run_btn = gr.Button("🚀 Run", scale=1)

    # ================================
    # 🔹 MIDDLE PANEL (CHART - LARGE)
    # ================================
    with gr.Row():
        output_plot = gr.Plot(
            label="📊 Market Dashboard",
            scale=1
        )

    # ================================
    # 🔹 BOTTOM PANEL (METRICS - LARGE)
    # ================================
    with gr.Row():
        output_text = gr.Textbox(
            label="📈 Metrics & Analysis",
            lines=12,   # 🔥 big box (no scroll)
            max_lines=20
        )

    # ================================
    # 🔄 INTERACTION
    # ================================
    run_btn.click(
        fn=run_with_loading,
        inputs=[mode, ticker],
        outputs=[output_text, output_plot],
        show_progress=True
    )

# ================================
# 🚀 LAUNCH
# ================================
demo.launch(debug=True, share=True)

/tmp/ipykernel_1318/1320016944.py:133: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_1318/1320016944.py:133: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fc3c4cfdd4dd3bf2ab.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipykernel_1318/3708243693.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="5y", progress=False)


Feature shape: torch.Size([500, 20, 9])
Feature shape: torch.Size([500, 20, 9])
Feature shape: torch.Size([500, 20, 9])
Feature shape: torch.Size([500, 20, 9])
Feature shape: torch.Size([500, 20, 9])


/tmp/ipykernel_1318/3708243693.py:2: FutureWarning:

YF.download() has changed argument auto_adjust default to True



...

/tmp/ipykernel_1318/3708243693.py:2: FutureWarning:

YF.download() has changed argument auto_adjust default to True



Feature shape: torch.Size([500, 20, 9])
Feature shape: torch.Size([500, 20, 9])
Feature shape: torch.Size([500, 20, 9])
Feature shape: torch.Size([500, 20, 9])
Feature shape: torch.Size([500, 20, 9])


/tmp/ipykernel_1318/3708243693.py:2: FutureWarning:

YF.download() has changed argument auto_adjust default to True



Feature shape: torch.Size([500, 20, 9])
Feature shape: torch.Size([500, 20, 9])
Feature shape: torch.Size([500, 20, 9])
Feature shape: torch.Size([500, 20, 9])


/tmp/ipykernel_1318/1155741844.py:3: FutureWarning:

YF.download() has changed argument auto_adjust default to True

